# 3c. Model Selection Consensus

**Purpose**: Do grid-search CV (3a) and SBI (3b) agree on which model
(BE vs SC) best describes each animal? If they disagree, which do we
trust and why?

**Protocol**:
1. Load results from 3a (grid-search) and 3b (SBI)
2. Align by animal — compare winner, error magnitudes, confidence
3. Identify discrepancies and diagnose causes
4. Final per-animal classification with confidence rating

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import pickle
import warnings
warnings.filterwarnings('ignore')

from behav_utils.plotting.styles import COLOURS, apply_style
apply_style()

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Configuration

In [ ]:
# Paths to results from 3a and 3b
GS_RESULTS_DIR = Path('../results/cv')           # grid-search pickles
SBI_RESULTS_DIR = Path('../results/sbi')          # SBI comparison results

# Or load from saved summary files
GS_CSV = GS_RESULTS_DIR / 'cv_comparison_anova.csv'   # from gather_cv_results.py
SBI_PKL = SBI_RESULTS_DIR / 'sbi_comparison_results.pkl'  # from 3b

ALPHA = 0.05  # significance threshold for ANOVA

## 2. Load Grid-Search Results

In [ ]:
# Load grid-search results
try:
    gs_df = pd.read_csv(GS_CSV)
    print(f"Grid-search: {len(gs_df)} animals")
    print(gs_df[['animal_id', 'be_mean', 'sc_mean', 'p_value', 'winner']].to_string(index=False))
    HAS_GS = True
except FileNotFoundError:
    print(f"Grid-search results not found at {GS_CSV}")
    print("Run 3a first, or update the path.")
    HAS_GS = False
    gs_df = pd.DataFrame()

## 3. Load SBI Results

In [ ]:
# Load SBI results
# Expected structure: dict with animal_id keys, each containing
# 'winner', 'p', 'be_mean', 'sc_mean' from run_animal_pipeline()
try:
    with open(SBI_PKL, 'rb') as f:
        sbi_raw = pickle.load(f)

    # Handle different possible formats
    if isinstance(sbi_raw, dict):
        sbi_rows = []
        for aid, res in sbi_raw.items():
            sbi_rows.append({
                'animal_id': aid if isinstance(aid, str) else res.get('animal_id', aid),
                'sbi_winner': res.get('winner', '?'),
                'sbi_p': res.get('p', np.nan),
                'sbi_be_mean': res.get('be_mean', np.nan),
                'sbi_sc_mean': res.get('sc_mean', np.nan),
            })
        sbi_df = pd.DataFrame(sbi_rows)
    elif isinstance(sbi_raw, list):
        sbi_df = pd.DataFrame(sbi_raw)
        sbi_df = sbi_df.rename(columns={
            'winner': 'sbi_winner', 'p': 'sbi_p',
            'be_mean': 'sbi_be_mean', 'sc_mean': 'sbi_sc_mean',
        })
    else:
        sbi_df = pd.DataFrame(sbi_raw)

    print(f"SBI: {len(sbi_df)} animals")
    print(sbi_df.to_string(index=False))
    HAS_SBI = True
except FileNotFoundError:
    print(f"SBI results not found at {SBI_PKL}")
    print("Run 3b first, or update the path.")
    HAS_SBI = False
    sbi_df = pd.DataFrame()

## 4. Merge & Compare

In [ ]:
if HAS_GS and HAS_SBI:
    # Rename GS columns for clarity
    gs_renamed = gs_df.rename(columns={
        'winner': 'gs_winner', 'p_value': 'gs_p',
        'be_mean': 'gs_be_mean', 'sc_mean': 'gs_sc_mean',
    })

    # Merge on animal_id
    merged = pd.merge(
        gs_renamed[['animal_id', 'gs_winner', 'gs_p', 'gs_be_mean', 'gs_sc_mean']],
        sbi_df[['animal_id', 'sbi_winner', 'sbi_p', 'sbi_be_mean', 'sbi_sc_mean']],
        on='animal_id', how='outer', indicator=True,
    )

    # Agreement
    both = merged[merged['_merge'] == 'both'].copy()
    both['agree'] = both['gs_winner'] == both['sbi_winner']

    # Confidence rating
    def confidence(row):
        if row['gs_winner'] == row['sbi_winner']:
            if row['gs_p'] < 0.01 and row['sbi_p'] < 0.01:
                return 'high'
            elif row['gs_p'] < ALPHA and row['sbi_p'] < ALPHA:
                return 'medium'
            else:
                return 'low'
        else:
            return 'disagreement'

    both['confidence'] = both.apply(confidence, axis=1)
    both['final_model'] = both.apply(
        lambda r: r['gs_winner'] if r['agree'] else 'Unresolved', axis=1,
    )

    print("Merged comparison:")
    display_cols = ['animal_id', 'gs_winner', 'gs_p', 'sbi_winner', 'sbi_p',
                    'agree', 'confidence', 'final_model']
    print(both[display_cols].to_string(index=False, float_format='{:.4f}'.format))

    n_agree = both['agree'].sum()
    n_total = len(both)
    print(f"\nAgreement: {n_agree}/{n_total} ({n_agree/n_total:.0%})")
    print(f"Confidence breakdown: {both['confidence'].value_counts().to_dict()}")

elif HAS_GS:
    print("Only grid-search results available.")
    both = gs_df.rename(columns={'winner': 'final_model'}).copy()
    both['confidence'] = both['p_value'].apply(
        lambda p: 'high' if p < 0.01 else ('medium' if p < ALPHA else 'low')
    )
elif HAS_SBI:
    print("Only SBI results available.")
    both = sbi_df.rename(columns={'sbi_winner': 'final_model'}).copy()
else:
    print("No results to compare. Run 3a and/or 3b first.")
    both = pd.DataFrame()

## 5. Visualisation

In [ ]:
if HAS_GS and HAS_SBI and len(both) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # ── Panel 1: Error comparison scatter ───────────────────────────────────
    ax = axes[0]
    # GS: BE-SC error difference
    both['gs_diff'] = both['gs_be_mean'] - both['gs_sc_mean']  # neg = BE wins
    both['sbi_diff'] = both['sbi_be_mean'] - both['sbi_sc_mean']

    colours = both['agree'].map({True: 'green', False: 'red'})
    ax.scatter(both['gs_diff'], both['sbi_diff'], c=colours, s=60, alpha=0.7,
               edgecolors='k', linewidth=0.5)
    for _, row in both.iterrows():
        ax.annotate(row['animal_id'], (row['gs_diff'], row['sbi_diff']),
                    fontsize=7, alpha=0.7)
    ax.axhline(0, color='grey', ls='--', alpha=0.3)
    ax.axvline(0, color='grey', ls='--', alpha=0.3)
    ax.set_xlabel('GS: BE−SC error (neg = BE wins)')
    ax.set_ylabel('SBI: BE−SC error (neg = BE wins)')
    ax.set_title('Error difference agreement')

    # ── Panel 2: Winner summary ────────────────────────────────────────────
    ax = axes[1]
    cats = ['BE', 'SC', 'Inconclusive', 'Unresolved']
    gs_counts = [len(both[both['gs_winner'] == c]) for c in cats]
    sbi_counts = [len(both[both['sbi_winner'] == c]) for c in cats]
    x = np.arange(len(cats))
    ax.bar(x - 0.15, gs_counts, 0.3, label='Grid-search', color=COLOURS.get('BE', 'steelblue'), alpha=0.6)
    ax.bar(x + 0.15, sbi_counts, 0.3, label='SBI', color=COLOURS.get('SC', 'darkorange'), alpha=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels(cats)
    ax.set_ylabel('Count')
    ax.set_title('Winner counts')
    ax.legend()

    # ── Panel 3: Confidence breakdown ──────────────────────────────────────
    ax = axes[2]
    conf_counts = both['confidence'].value_counts()
    conf_colours = {'high': 'green', 'medium': 'gold', 'low': 'orange', 'disagreement': 'red'}
    bars = ax.bar(conf_counts.index, conf_counts.values,
                  color=[conf_colours.get(c, 'grey') for c in conf_counts.index])
    ax.set_ylabel('Count')
    ax.set_title('Classification confidence')

    fig.suptitle('3c. Model Selection Consensus', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 6. Diagnose Disagreements

In [ ]:
if HAS_GS and HAS_SBI and len(both) > 0:
    disagree = both[~both['agree']]
    if len(disagree) > 0:
        print(f"\n{len(disagree)} disagreements to investigate:\n")
        for _, row in disagree.iterrows():
            print(f"  {row['animal_id']}:")
            print(f"    GS:  {row['gs_winner']} (BE={row['gs_be_mean']:.5f}, "
                  f"SC={row['gs_sc_mean']:.5f}, p={row['gs_p']:.3g})")
            print(f"    SBI: {row['sbi_winner']} (BE={row['sbi_be_mean']:.5f}, "
                  f"SC={row['sbi_sc_mean']:.5f}, p={row['sbi_p']:.3g})")
            gs_margin = abs(row['gs_be_mean'] - row['gs_sc_mean'])
            sbi_margin = abs(row['sbi_be_mean'] - row['sbi_sc_mean'])
            print(f"    GS margin: {gs_margin:.5f}, SBI margin: {sbi_margin:.5f}")
            if gs_margin < 0.001 or sbi_margin < 0.001:
                print(f"    → Very small margin — borderline animal")
            print()
    else:
        print("Perfect agreement across all animals.")

### 6b. Visual Disagreement Diagnostics

For each disagreeing animal, show the actual update matrix alongside
the best-fit model predictions from each method. This reveals *why*
the methods disagree — different sensitivity to specific UM features.

In [ ]:
# Load real data for detailed comparison
from behav_utils.data.loading import load_experiment
from behav_utils.data.selection import select_sessions
from behav_utils.analysis.update_matrix import (
    compute_update_matrix, compute_update_matrix_from_sessions,
)
from behav_utils.plotting.update_matrix import (
    plot_update_matrix, plot_phase_update_matrices,
)
from behav_utils.plotting.psychometric import plot_psychometric_overlay

CONFIG_PATH = Path('../config.yaml')
STAGE = 'Full_Task_Cont'

try:
    experiment = load_experiment(CONFIG_PATH)
    HAS_RAW = True
    print(f'Loaded {len(experiment.animal_ids)} animals for diagnostics')
except Exception as e:
    HAS_RAW = False
    print(f'Could not load raw data: {e}')

In [ ]:
if HAS_GS and HAS_SBI and HAS_RAW and len(both) > 0:
    disagree = both[~both['agree']]

    if len(disagree) == 0:
        print('No disagreements — all animals classified consistently.')
    else:
        for _, row in disagree.iterrows():
            aid = row['animal_id']
            print(f"\n{'='*60}")
            print(f"  {aid}: GS→{row['gs_winner']}, SBI→{row['sbi_winner']}")
            print(f"  GS margins: BE={row['gs_be_mean']:.5f}, SC={row['gs_sc_mean']:.5f}")
            print(f"  SBI margins: BE={row['sbi_be_mean']:.5f}, SC={row['sbi_sc_mean']:.5f}")

            try:
                animal = experiment.get_animal(aid)
            except KeyError:
                print(f'  Animal {aid} not found in data — skipping')
                continue

            # Get expert sessions
            expert = select_sessions(
                animal, stage=STAGE, distribution='Uniform',
                min_accuracy=0.70, last_fraction=0.50,
            )
            if len(expert) < 3:
                print(f'  Too few expert sessions ({len(expert)}) — skipping')
                continue

            # ── 1. Data update matrix ────────────────────────────────────
            um_data, cm_data, info_data = compute_update_matrix_from_sessions(
                expert, method='pool',
            )

            fig, ax = plt.subplots(1, 1, figsize=(5, 4.5))
            plot_update_matrix(
                um_data, ax=ax,
                title=f'{aid} — Empirical UM ({info_data["n_sessions"]}s pooled)',
            )
            plt.tight_layout()
            plt.show()

            # ── 2. Psychometric curve ────────────────────────────────────
            fig, ax, psych_info = plot_psychometric_overlay(
                {'Expert Uniform': expert},
                mode='pooled', n_bootstrap=200,
                title=f'{aid} — Expert psychometric',
            )
            plt.show()

            # ── 3. Interpretation ────────────────────────────────────────
            # Check for ambiguous UM features
            profile = np.nanmean(um_data, axis=0)
            profile_range = np.nanmax(profile) - np.nanmin(profile)
            n_valid_cols = np.sum(~np.all(np.isnan(um_data), axis=0))

            print(f'  UM profile range: {profile_range:.4f}')
            print(f'  Valid UM columns: {n_valid_cols}/8')

            if profile_range < 0.02:
                print(f'  → Very flat profile — weak serial dependence. '
                      f'Neither model fits well; classification unreliable.')
            if n_valid_cols < 6:
                print(f'  → Many NaN columns — insufficient data in '
                      f'extreme bins. MSE comparison unreliable.')

            gs_margin = abs(row['gs_be_mean'] - row['gs_sc_mean'])
            sbi_margin = abs(row['sbi_be_mean'] - row['sbi_sc_mean'])
            if gs_margin < 0.001 and sbi_margin < 0.001:
                print(f'  → Both methods show tiny margins — '
                      f'genuinely ambiguous animal.')
            elif gs_margin < 0.001:
                print(f'  → GS margin very small — SBI classification '
                      f'more decisive, prefer SBI.')
            elif sbi_margin < 0.001:
                print(f'  → SBI margin very small — GS classification '
                      f'more decisive, prefer GS.')

elif not HAS_RAW:
    print('Raw data not available — text diagnostics only (see cell above).')

## 7. Final Classification Table

In [ ]:
if len(both) > 0:
    final = both[['animal_id', 'final_model', 'confidence']].copy()
    print("Final model assignments:")
    print(final.to_string(index=False))

    # Save
    # results_dir = Path('../results')
    # results_dir.mkdir(parents=True, exist_ok=True)
    # final.to_csv(results_dir / 'model_classification_consensus.csv', index=False)
    # print(f"\nSaved to {results_dir / 'model_classification_consensus.csv'}")

## Interpretation

**High agreement**: Both methods robustly classify the same animals the same way.
This is the best case — the classification is method-independent.

**Disagreements on borderline animals**: Small error margins in both methods.
These animals may genuinely lie between the two model classes, or the data
may be insufficient for classification. Report as 'Unresolved'.

**Systematic disagreement**: One method consistently picks BE where the other
picks SC. This would indicate a methodological bias — investigate whether
the SBI stat set or the grid-search resolution is driving the difference.

**Note**: The SBI results should be from the bug-fixed `cv_um_comparison`
(block-level splitting, not trial-level). If using pre-fix results, re-run
3b first.